[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DavinciDreams/SymbioGPT/blob/main/compress_juliaslm.ipynb)

# Compression via Downscaling Projection (Reverse Symbiogenesis)

Compress **JuliaSLM** (5.04M params, d=256, 6L, val_loss=3.54) into smaller architectures
using weight truncation — the inverse of the symbiogenesis projection fusion.

Instead of zero-padding UP (d=256→512), we **truncate DOWN** (d=256→192) by slicing
the first `d_target` dimensions from every weight matrix, keeping the first `n_target`
heads, and dropping layers from the end. This is one-shot (no evolutionary search) —
just array slicing, then fine-tune to recover.

**Three target configs** (all keep head_dim=64 for RoPE, vocab=2000, ctx=256):

| Config | d_model | Layers | Heads | FFN | Params | Reduction |
|--------|---------|--------|-------|-----|--------|-----------|
| A (3M) | 192 | 6 | 3×64 | 480 | ~2.93M | 42% |
| B (2.5M) | 192 | 5 | 3×64 | 480 | ~2.51M | 50% |
| C (2M) | 192 | 4 | 3×64 | 480 | ~2.08M | 59% |

**Why this works**: The scaling law data shows all models on this corpus converge to
~3.5-3.7 val_loss regardless of size (5M–11M). If the corpus can't distinguish
5M from 3M, we get ~40% free compression. If it can, we map the Pareto frontier.

GitHub: https://github.com/DavinciDreams/SymbioGPT

In [ ]:
# 1. Setup
!pip install -q wandb huggingface_hub
!mkdir -p /content/SymbioGPT
%cd /content/SymbioGPT

In [ ]:
# 2. GPU check
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    print(f"Memory: {mem / 1e9:.1f} GB")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 3. W&B + HF login
import os
import wandb
from huggingface_hub import login as hf_login

# Prefer Colab secrets, fall back to env vars, then interactive prompt
try:
    from google.colab import userdata
    os.environ.setdefault("WANDB_API_KEY", userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except (ImportError, Exception):
    pass  # Not in Colab or secrets not configured

wandb.login()
hf_login(token=os.environ.get("HF_TOKEN"), add_to_git_credential=False)

In [ ]:
# 4. Download data + JuliaSLM weights
import os, sys, math, time, copy
from dataclasses import dataclass
from typing import Dict, List, Tuple
import numpy as np
from huggingface_hub import hf_hub_download
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_REPO = "LisaMegaWatts/SymbioGPT-10M"
SLM_REPO = "LisaMegaWatts/JuliaSLM"
os.makedirs("data", exist_ok=True)

print("Downloading pre-tokenized data...")
hf_hub_download(repo_id=DATA_REPO, filename="data/train_curated.txt.tokens.pt", local_dir=".")
hf_hub_download(repo_id=DATA_REPO, filename="data/val.txt.tokens.pt", local_dir=".")

print("Downloading JuliaSLM weights (NPZ)...")
hf_hub_download(repo_id=SLM_REPO, filename="juliaslm_weights.npz", local_dir=".")

CTX = 256
print("Loading tokens...")
train_tokens = torch.load("data/train_curated.txt.tokens.pt", weights_only=True).tolist()
val_tokens = torch.load("data/val.txt.tokens.pt", weights_only=True).tolist()

def chunk(tokens, seq_len):
    n = len(tokens) // (seq_len + 1)
    tokens = tokens[:n * (seq_len + 1)]
    data = torch.tensor(tokens, dtype=torch.long).reshape(n, seq_len + 1)
    return data[:, :-1], data[:, 1:]

train_inputs, train_labels = chunk(train_tokens, CTX)
val_inputs, val_labels = chunk(val_tokens, CTX)
print(f"Train: {len(train_inputs):,} seqs ({len(train_inputs)*CTX:,} tokens)")
print(f"Val: {len(val_inputs):,} seqs")
del train_tokens, val_tokens

In [ ]:
# 5. JuliaSLM model definition
#
# Standard LLaMA-style: MHA (separate Q/K/V/O), RMSNorm, SwiGLU, RoPE.
# Same class used for both source (d=256) and compressed targets (d=192).
#
# IMPORTANT: Julia's column-major reshape(result, HD, T, H, B) fills dims
# HD→T→H→B (fastest→slowest). Python's row-major equivalent is
# view(B, H, T, HD) — NOT view(B, T, H, HD).transpose(1, 2).
# The model was TRAINED with Julia's layout, so we must match it exactly.

@dataclass
class JuliaSLMConfig:
    d_model: int = 256
    n_layers: int = 6
    n_heads: int = 4
    head_dim: int = 64
    ffn_inner: int = 640
    context_length: int = 256
    vocab_size: int = 2000
    weight_tying: bool = True
    rope_base: float = 10000.0


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 256, base: float = 10000.0):
        super().__init__()
        freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        positions = torch.arange(max_seq_len).float()
        angles = torch.outer(positions, freqs)
        self.register_buffer("cos_cache", angles.cos())
        self.register_buffer("sin_cache", angles.sin())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(2)
        half = x.size(-1) // 2
        x1, x2 = x[..., :half], x[..., half:]
        cos = self.cos_cache[:seq_len, :half].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cache[:seq_len, :half].unsqueeze(0).unsqueeze(0)
        return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, head_dim: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        total_dim = n_heads * head_dim
        self.wq = nn.Linear(d_model, total_dim, bias=False)
        self.wk = nn.Linear(d_model, total_dim, bias=False)
        self.wv = nn.Linear(d_model, total_dim, bias=False)
        self.wo = nn.Linear(total_dim, d_model, bias=False)

    def forward(self, x: torch.Tensor, rope: RotaryEmbedding,
                mask: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        H, HD = self.n_heads, self.head_dim
        # Julia-matching reshape: view(B, H, T, HD) matches Julia's
        # column-major reshape(HD, T, H, B)
        q = self.wq(x).view(B, H, T, HD)
        k = self.wk(x).view(B, H, T, HD)
        v = self.wv(x).view(B, H, T, HD)
        q = rope(q)
        k = rope(k)
        scale = 1.0 / math.sqrt(HD)
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn = attn + mask
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        out = out.contiguous().view(B, T, H * HD)
        return self.wo(out)


class SwiGLUFFN(nn.Module):
    def __init__(self, d_model: int, inner_dim: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, inner_dim, bias=False)
        self.v = nn.Linear(d_model, inner_dim, bias=False)
        self.w2 = nn.Linear(inner_dim, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.v(x))


class TransformerBlock(nn.Module):
    def __init__(self, config: JuliaSLMConfig):
        super().__init__()
        self.ln1 = RMSNorm(config.d_model)
        self.attn = CausalSelfAttention(config.d_model, config.n_heads, config.head_dim)
        self.ln2 = RMSNorm(config.d_model)
        self.ffn = SwiGLUFFN(config.d_model, config.ffn_inner)

    def forward(self, x: torch.Tensor, rope: RotaryEmbedding,
                mask: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x), rope, mask)
        x = x + self.ffn(self.ln2(x))
        return x


class JuliaSLM(nn.Module):
    def __init__(self, config: JuliaSLMConfig):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.rope = RotaryEmbedding(config.head_dim, config.context_length, config.rope_base)
        self.blocks = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config.n_layers)]
        )
        self.ln_f = RMSNorm(config.d_model)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        mask = torch.triu(
            torch.full((T, T), float("-inf"), device=x.device, dtype=x.dtype),
            diagonal=1,
        )
        for block in self.blocks:
            x = block(x, self.rope, mask)
        x = self.ln_f(x)
        return F.linear(x, self.tok_emb.weight)


source_config = JuliaSLMConfig()
print(f"Source: d={source_config.d_model}, L={source_config.n_layers}, "
      f"H={source_config.n_heads}, hd={source_config.head_dim}, ffn={source_config.ffn_inner}")

In [ ]:
# 6. Load JuliaSLM weights from NPZ + verify baseline

npz_data = np.load("juliaslm_weights.npz")

print("NPZ keys:")
for key in sorted(npz_data.files):
    arr = npz_data[key]
    print(f"  {key}: {arr.shape} {arr.dtype}")

source_model = JuliaSLM(source_config).to(device)

# Map NPZ keys to PyTorch state_dict keys.
# NPZ stores bare names ("blocks.0.attn.wq"), nn.Linear expects ".weight" suffix.
model_keys = set(source_model.state_dict().keys())
state_dict = {}
for key in npz_data.files:
    if key.startswith("_hp_"):
        continue
    tensor = torch.from_numpy(npz_data[key].copy())
    if key in model_keys:
        state_dict[key] = tensor
    elif key + ".weight" in model_keys:
        state_dict[key + ".weight"] = tensor
    else:
        print(f"  WARNING: no match for NPZ key '{key}'")

missing, unexpected = source_model.load_state_dict(state_dict, strict=False)
print(f"\nLoaded {len(state_dict)} arrays")
rope_missing = [k for k in missing if "rope" in k or "cache" in k]
other_missing = [k for k in missing if k not in rope_missing]
if rope_missing:
    print(f"Missing (expected, RoPE buffers): {rope_missing}")
if other_missing:
    print(f"WARNING — Missing weights: {other_missing}")

source_model.eval()
n_source_params = sum(p.numel() for p in source_model.parameters())
print(f"Source model: {n_source_params:,} params ({n_source_params/1e6:.2f}M)")


def evaluate_model(model, val_inputs, val_labels, batch_size=64):
    """Compute val loss and perplexity."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    dev = next(model.parameters()).device
    with torch.no_grad():
        for i in range(0, len(val_inputs), batch_size):
            batch_in = val_inputs[i:i+batch_size].to(dev)
            batch_tgt = val_labels[i:i+batch_size].to(dev)
            logits = model(batch_in)
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.float().reshape(B*T, V), batch_tgt.reshape(B*T), reduction="sum"
            )
            total_loss += loss.item()
            total_tokens += B * T
    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 20.0))
    return avg_loss, ppl


print("\nEvaluating source model baseline...")
source_loss, source_ppl = evaluate_model(source_model, val_inputs, val_labels)
print(f"Source JuliaSLM: val_loss={source_loss:.4f} ppl={source_ppl:.1f}")
print(f"Expected:        val_loss=3.5403  ppl=34.5")

In [ ]:
# 7. Downscaling projection function
#
# The inverse of symbiogenesis fusion: truncate weights instead of zero-padding.
# - Embedding: keep first d_target cols
# - Attention Q/K/V: keep first n_heads_target * head_dim rows, first d_target cols
# - Attention O: keep first d_target rows, first n_heads_target * head_dim cols
# - FFN gate/up: keep first ffn_target rows, first d_target cols
# - FFN down: keep first d_target rows, first ffn_target cols
# - RMSNorm: keep first d_target entries
# - Layers: keep first n_layers_target layers (drop from end)

def project_down(source_model, target_config):
    """Downscale JuliaSLM weights by truncation into a smaller architecture."""
    src_cfg = source_model.config
    src_sd = source_model.state_dict()
    
    d_s = src_cfg.d_model      # 256
    d_t = target_config.d_model  # 192
    h_s = src_cfg.n_heads       # 4
    h_t = target_config.n_heads  # 3
    hd = src_cfg.head_dim       # 64 (same for both)
    ffn_s = src_cfg.ffn_inner   # 640
    ffn_t = target_config.ffn_inner  # 480
    n_layers = target_config.n_layers  # 4, 5, or 6
    
    target = JuliaSLM(target_config)
    td = target.state_dict()
    
    # Embedding: (vocab, d_s) → (vocab, d_t)
    td['tok_emb.weight'] = src_sd['tok_emb.weight'][:, :d_t].clone()
    
    # Per layer: keep first n_layers layers, truncate all weight dims
    for i in range(n_layers):
        pfx = f'blocks.{i}'
        
        # Attention: keep first h_t heads (each head is hd=64 wide)
        td[f'{pfx}.attn.wq.weight'] = src_sd[f'{pfx}.attn.wq.weight'][:h_t*hd, :d_t].clone()
        td[f'{pfx}.attn.wk.weight'] = src_sd[f'{pfx}.attn.wk.weight'][:h_t*hd, :d_t].clone()
        td[f'{pfx}.attn.wv.weight'] = src_sd[f'{pfx}.attn.wv.weight'][:h_t*hd, :d_t].clone()
        td[f'{pfx}.attn.wo.weight'] = src_sd[f'{pfx}.attn.wo.weight'][:d_t, :h_t*hd].clone()
        
        # FFN: truncate inner dim
        td[f'{pfx}.ffn.w1.weight'] = src_sd[f'{pfx}.ffn.w1.weight'][:ffn_t, :d_t].clone()
        td[f'{pfx}.ffn.v.weight'] = src_sd[f'{pfx}.ffn.v.weight'][:ffn_t, :d_t].clone()
        td[f'{pfx}.ffn.w2.weight'] = src_sd[f'{pfx}.ffn.w2.weight'][:d_t, :ffn_t].clone()
        
        # RMSNorm: keep first d_t entries
        td[f'{pfx}.ln1.weight'] = src_sd[f'{pfx}.ln1.weight'][:d_t].clone()
        td[f'{pfx}.ln2.weight'] = src_sd[f'{pfx}.ln2.weight'][:d_t].clone()
    
    # Final norm
    td['ln_f.weight'] = src_sd['ln_f.weight'][:d_t].clone()
    
    # Load into target model (strict=False for RoPE buffers)
    target.load_state_dict(td, strict=False)
    return target


print("project_down() defined.")
print(f"Source: d={source_config.d_model}, {source_config.n_layers}L, "
      f"{source_config.n_heads}H, ffn={source_config.ffn_inner}")

In [ ]:
# 8. Create 3 compressed configs, project, and evaluate pre-fine-tune

configs = {
    "A-3M": JuliaSLMConfig(d_model=192, n_layers=6, n_heads=3, head_dim=64,
                            ffn_inner=480, context_length=256, vocab_size=2000),
    "B-2.5M": JuliaSLMConfig(d_model=192, n_layers=5, n_heads=3, head_dim=64,
                              ffn_inner=480, context_length=256, vocab_size=2000),
    "C-2M": JuliaSLMConfig(d_model=192, n_layers=4, n_heads=3, head_dim=64,
                            ffn_inner=480, context_length=256, vocab_size=2000),
}

compressed_models = {}
pre_ft_results = {}

print(f"{'Config':<10} {'Params':>10} {'Reduction':>10} {'Pre-FT Loss':>12} {'Pre-FT PPL':>11}")
print("-" * 55)

for name, cfg in configs.items():
    model_c = project_down(source_model, cfg)
    model_c = model_c.to(device)
    n_params = sum(p.numel() for p in model_c.parameters())
    reduction = 1 - n_params / n_source_params
    
    loss, ppl = evaluate_model(model_c, val_inputs, val_labels)
    pre_ft_results[name] = {"params": n_params, "reduction": reduction,
                             "pre_loss": loss, "pre_ppl": ppl}
    compressed_models[name] = model_c
    
    print(f"{name:<10} {n_params:>10,} {reduction:>9.1%} {loss:>12.4f} {ppl:>11.1f}")

print(f"\n{'Source':<10} {n_source_params:>10,} {'---':>10} {source_loss:>12.4f} {source_ppl:>11.1f}")
print(f"\nAll compressed models created. Pre-fine-tune losses should be degraded")
print(f"but NOT random-init level (~7.6). If they're close to source, the")
print(f"truncated dimensions carried minimal information.")

In [ ]:
# 9. Fine-tune function (shared across all configs)

def finetune(model, config_name, config, train_inputs, train_labels,
             val_inputs, val_labels, n_steps=2000, lr=6e-4, batch_size=64,
             warmup=200, eval_every=250):
    """Fine-tune a compressed model. Returns (model, best_loss, best_ppl, history)."""
    
    n_params = sum(p.numel() for p in model.parameters())
    pre_loss, pre_ppl = evaluate_model(model, val_inputs, val_labels)
    
    run = wandb.init(
        project="symbiogenesis",
        name=f"compress-juliaslm-{config_name}",
        config={
            "method": "downscaling_projection",
            "source": "JuliaSLM (5.04M, d=256, 6L, val_loss=3.54)",
            "config_name": config_name,
            "d_model": config.d_model,
            "n_layers": config.n_layers,
            "n_heads": config.n_heads,
            "head_dim": config.head_dim,
            "ffn_inner": config.ffn_inner,
            "n_params": n_params,
            "pre_ft_loss": pre_loss,
            "n_steps": n_steps,
            "lr": lr,
            "batch_size": batch_size,
            "warmup": warmup,
        },
        tags=["compression", "downscaling", "juliaslm", "symbiogenesis", config_name],
        reinit="finish_previous",
    )
    
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=0.1, betas=(0.9, 0.95)
    )
    min_lr = lr * 0.1
    
    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / max(warmup, 1)
        progress = (step - warmup) / max(n_steps - warmup, 1)
        return max(min_lr / lr, 0.5 * (1.0 + math.cos(math.pi * progress)))
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))
    
    n_train = len(train_inputs)
    model.train()
    best_loss = float("inf")
    best_ppl = float("inf")
    history = []
    t_start = time.time()
    step = 0
    
    print(f"\n{'='*55}")
    print(f"Fine-tuning {config_name}: {n_params:,} params, {n_steps} steps")
    print(f"Pre-FT: loss={pre_loss:.4f} ppl={pre_ppl:.1f}")
    print(f"Precision: {amp_dtype}, batch={batch_size}, lr={lr}")
    print(f"{'='*55}")
    
    while step < n_steps:
        perm = torch.randperm(n_train)
        for i in range(0, n_train, batch_size):
            if step >= n_steps:
                break
            
            idx = perm[i:i+batch_size]
            batch_in = train_inputs[idx].to(device)
            batch_tgt = train_labels[idx].to(device)
            
            with torch.amp.autocast("cuda", enabled=True, dtype=amp_dtype):
                logits = model(batch_in)
                B, T, V = logits.shape
                loss = F.cross_entropy(logits.reshape(B*T, V), batch_tgt.reshape(B*T))
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
            if step % 50 == 0:
                wandb.log({"train/loss": loss.item(),
                           "train/lr": scheduler.get_last_lr()[0]}, step=step)
            
            if step > 0 and step % eval_every == 0:
                val_loss, val_ppl = evaluate_model(model, val_inputs, val_labels)
                history.append((step, val_loss, val_ppl))
                wandb.log({"val/loss": val_loss, "val/ppl": val_ppl}, step=step)
                marker = " ** BEST **" if val_loss < best_loss else ""
                if val_loss < best_loss:
                    best_loss = val_loss
                    best_ppl = val_ppl
                    torch.save(model.state_dict(), f"compressed_{config_name}_best.pt")
                elapsed = time.time() - t_start
                print(f"  [{config_name} step {step:5d}] val={val_loss:.4f} ppl={val_ppl:.1f} "
                      f"lr={scheduler.get_last_lr()[0]:.2e} ({elapsed:.0f}s){marker}")
                model.train()
            
            step += 1
    
    # Final eval
    final_loss, final_ppl = evaluate_model(model, val_inputs, val_labels)
    history.append((step, final_loss, final_ppl))
    if final_loss < best_loss:
        best_loss = final_loss
        best_ppl = final_ppl
        torch.save(model.state_dict(), f"compressed_{config_name}_best.pt")
    
    elapsed = time.time() - t_start
    wandb.log({"val/final_loss": best_loss, "val/final_ppl": best_ppl})
    print(f"  {config_name} done ({elapsed:.0f}s): best val_loss={best_loss:.4f} ppl={best_ppl:.1f}")
    wandb.finish()
    
    # Reload best checkpoint
    best_sd = torch.load(f"compressed_{config_name}_best.pt", map_location=device, weights_only=True)
    model.load_state_dict(best_sd, strict=False)
    model.eval()
    
    return model, best_loss, best_ppl, history


print("finetune() defined.")

In [ ]:
# 10. Fine-tune all 3 compressed models sequentially

post_ft_results = {}

for name in ["A-3M", "B-2.5M", "C-2M"]:
    model_c = compressed_models[name]
    cfg = configs[name]
    
    model_c, best_loss, best_ppl, history = finetune(
        model_c, name, cfg,
        train_inputs, train_labels,
        val_inputs, val_labels,
        n_steps=2000, lr=6e-4, batch_size=64,
        warmup=200, eval_every=250,
    )
    
    post_ft_results[name] = {
        "best_loss": best_loss,
        "best_ppl": best_ppl,
        "history": history,
    }
    compressed_models[name] = model_c

print("\nAll fine-tuning complete!")

In [ ]:
# 11. Results comparison

print("\n" + "=" * 75)
print("DOWNSCALING PROJECTION RESULTS — JuliaSLM Compression")
print("=" * 75)

print(f"\n{'Config':<12} {'Params':>10} {'Reduc':>7} {'Pre-FT':>9} {'Post-FT':>9} {'Recovery':>10} {'PPL':>8}")
print("-" * 67)

for name in ["A-3M", "B-2.5M", "C-2M"]:
    pre = pre_ft_results[name]
    post = post_ft_results[name]
    recovery = pre["pre_loss"] - post["best_loss"]
    print(f"{name:<12} {pre['params']:>10,} {pre['reduction']:>6.1%} "
          f"{pre['pre_loss']:>9.4f} {post['best_loss']:>9.4f} "
          f"{recovery:>+9.4f} {post['best_ppl']:>8.1f}")

print(f"{'Source':<12} {n_source_params:>10,} {'---':>7} "
      f"{source_loss:>9.4f} {'---':>9} {'---':>10} {source_ppl:>8.1f}")

# Scaling law context
print(f"\n{'='*75}")
print("SCALING LAW CONTEXT (curated data, BPE vocab=2000, ctx=256)")
print(f"{'='*75}")
print(f"\n{'Model':<30} {'Params':>10} {'Val Loss':>10} {'PPL':>8}")
print("-" * 60)

# Add compressed models
for name in ["C-2M", "B-2.5M", "A-3M"]:
    pre = pre_ft_results[name]
    post = post_ft_results[name]
    print(f"{'JuliaSLM-'+name:<30} {pre['params']/1e6:>9.2f}M {post['best_loss']:>10.4f} {post['best_ppl']:>8.1f}")

# Known baselines
print(f"{'SymbioSLM':<30} {'4.07M':>10} {'3.6200':>10} {'37.3':>8}")
print(f"{'MonarchSLM':<30} {'4.98M':>10} {'3.6500':>10} {'38.4':>8}")
print(f"{'JuliaSLM (source)':<30} {'5.04M':>10} {source_loss:>10.4f} {source_ppl:>8.1f}")
print(f"{'SymbioGPT-10M':<30} {'11.05M':>10} {'3.5630':>10} {'35.3':>8}")

# Find best compressed config
best_name = min(post_ft_results, key=lambda n: post_ft_results[n]["best_loss"])
best = post_ft_results[best_name]
best_pre = pre_ft_results[best_name]
print(f"\nBest compressed: {best_name} ({best_pre['params']/1e6:.2f}M) "
      f"val_loss={best['best_loss']:.4f} ppl={best['best_ppl']:.1f} "
      f"({best_pre['reduction']:.0%} smaller than source)")

In [ ]:
# 12. Upload best compressed model to HuggingFace
from huggingface_hub import HfApi, create_repo
import json

best_name = min(post_ft_results, key=lambda n: post_ft_results[n]["best_loss"])
best_post = post_ft_results[best_name]
best_pre = pre_ft_results[best_name]
best_cfg = configs[best_name]

COMPRESSED_REPO = "LisaMegaWatts/JuliaSLM-compressed"

# Save metadata
os.makedirs("compressed", exist_ok=True)
meta_path = "compressed/compression_metadata.json"
all_results = {}
for name in ["A-3M", "B-2.5M", "C-2M"]:
    pre = pre_ft_results[name]
    post = post_ft_results[name]
    cfg = configs[name]
    all_results[name] = {
        "config": {
            "d_model": cfg.d_model, "n_layers": cfg.n_layers,
            "n_heads": cfg.n_heads, "head_dim": cfg.head_dim,
            "ffn_inner": cfg.ffn_inner, "context_length": cfg.context_length,
            "vocab_size": cfg.vocab_size,
        },
        "params": pre["params"],
        "reduction": pre["reduction"],
        "pre_finetune_loss": pre["pre_loss"],
        "post_finetune_loss": post["best_loss"],
        "post_finetune_ppl": post["best_ppl"],
    }

with open(meta_path, "w") as f:
    json.dump({
        "method": "downscaling_projection",
        "source_model": "LisaMegaWatts/JuliaSLM",
        "source_params": n_source_params,
        "source_loss": source_loss,
        "best_config": best_name,
        "finetune_steps": 2000,
        "finetune_lr": 6e-4,
        "configs": all_results,
    }, f, indent=2)

print(f"Saved metadata: {meta_path}")

# Upload best checkpoint + all checkpoints + metadata
hf_api = HfApi()
try:
    create_repo(COMPRESSED_REPO, exist_ok=True)
    
    # Upload all config checkpoints
    for name in ["A-3M", "B-2.5M", "C-2M"]:
        ckpt = f"compressed_{name}_best.pt"
        if os.path.exists(ckpt):
            size_mb = os.path.getsize(ckpt) / 1e6
            post = post_ft_results[name]
            print(f"Uploading {ckpt} ({size_mb:.1f} MB, loss={post['best_loss']:.4f})...")
            hf_api.upload_file(
                path_or_fileobj=ckpt,
                path_in_repo=ckpt,
                repo_id=COMPRESSED_REPO,
                commit_message=f"{name}: val_loss={post['best_loss']:.4f} ppl={post['best_ppl']:.1f}",
            )
    
    # Upload metadata
    hf_api.upload_file(
        path_or_fileobj=meta_path,
        path_in_repo="compression_metadata.json",
        repo_id=COMPRESSED_REPO,
        commit_message="Compression metadata and results",
    )
    
    print(f"\nUploaded to: https://huggingface.co/{COMPRESSED_REPO}")
except Exception as e:
    print(f"HF upload failed: {e}")

print("\nDone!")